# DVC "Git dla danych i modeli"
Ten notebook prezentuje minimalny workflow wersjonowania danych i modelu.


## Instalacja bibliotek

In [1]:
! pip install dvc scikit-learn pandas joblib
! git init
! dvc init

  Using cached orjson-3.11.9-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (41 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached orjson-3.11.9-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 3.7 MB/s  0:00:01 eta 0:00:01
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=005998b1c5247563cc7a285aee8029d9cb8904e0388c11a05da419feb4a3aab3
  Stored in directory: /home/mzar/.cache/pip/wheels/d5/b3/74/a35b66048c9de6631cd74cbc9475e6feb3e69a467983446bd8
Successfully built antlr4-python3-runtime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44/44 [dvc] [dvc] [celery]ore]]
hint: Using 'master' as the name for the initial branch. This default branch name


## Import bibliotek i zbioru danych

In [4]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

Path("data").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

iris = load_iris(as_frame=True)
df = iris.frame
df.to_csv("data/iris.csv", index=False)
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Dodanie danych do DVC

W terminalu:

```bash
dvc add data/iris.csv
git add data/iris.csv.dvc data/.gitignore .dvc .dvcignore
git commit -m "Add Iris dataset tracked by DVC"
```

In [9]:
! dvc add data/iris.csv

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/mzar/mlops-practical-examples/notebooks/pl/.dvc/ca
                                                                                
!
  0%|          |Checking out /home/mzar/mlops-practica0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 49.53file/s]

To track the changes with git, run:

	git add data/iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [10]:
! git add data/iris.csv.dvc data/.gitignore .dvc .dvcignore

In [11]:
! git commit -m "Add Iris dataset tracked by DVC"

[master (root-commit) 5abe28e] Add Iris dataset tracked by DVC
 5 files changed, 12 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 data/.gitignore
 create mode 100644 data/iris.csv.dvc


### Zdalne repozytorium
(w tym przykłaadzie lokalne)

```bash
mkdir -p /tmp/dvc-storage
dvc remote add -d localstorage /tmp/dvc-storage
dvc push
```

In [12]:
! mkdir -p /tmp/dvc-storage
! dvc remote add -d localstorage /tmp/dvc-storage
! dvc push

Setting 'localstorage' as a default remote.
Pushing
!
  0% Checking cache in '/tmp/dvc-storage/files/md5'| |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/home/mzar/mlops-practical-examples/notebooks/pl/.dvc/ca
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing                                                                         
1 file pushed


## Trenowanie i utworzenie wersji modelu

In [13]:
df = pd.read_csv("data/iris.csv")
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print({"accuracy": acc})

joblib.dump({"model": model, "accuracy": acc, "version": "v1"}, "models/iris_rf.joblib")


{'accuracy': 0.9}


['models/iris_rf.joblib']

## Wersjonowanie modelu

W terminalu:

```bash
dvc add models/iris_rf.joblib
git add models/iris_rf.joblib.dvc models/.gitignore
git commit -m "Add first RF model version"
dvc push
```

In [14]:
! dvc add models/iris_rf.joblib
! git add models/iris_rf.joblib.dvc models/.gitignore
! git commit -m "Add first RF model version"
! dvc push

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/mzar/mlops-practical-examples/notebooks/pl/.dvc/ca
                                                                                
!
  0%|          |Adding models/iris_rf.joblib to cache 0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/mzar/mlops-practica0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 41.35file/s]

To track the changes with git, run:

	git add models/.gitignore models/iris_rf.joblib.dvc

To enable auto staging, run:

	dvc config core.autostage true
[master 6ec5d87] Add first RF model version
 2 files changed, 6 insertions(+)
 create mode 100644 models/.gitignore
 create mode 100644 models/iris_rf.joblib.dvc
Pushing
!
 

Następnie zmień parametry modelu, np. `n_estimators=200`, ponownie wytrenuj model i wykonaj:

```bash
git add models/iris_rf.joblib.dvc
git commit -m "Update RF model version"
dvc push
```

In [ ]:
! git add models/iris_rf.joblib.dvc
! git commit -m "Update RF model version"
! dvc push

### Podsumowanie
- Git trzyma kod i lekkie metadane DVC.
- DVC trzyma duże dane i modele w storage.
- Wersja modelu bez wersji danych jest niepełna.
- Możemy wrócić do starego commita i wykonać `dvc pull`, żeby odtworzyć dokładny stan danych/modelu.
